# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/3bud-ZC/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Source reviewed: **FlyRank — The State of AI-Driven SEO in Numbers (2026)**.

### Finding A — growing vs declining content
The paper reports that growing pages averaged about **185 days old** while declining pages averaged about **228 days**, while average word count was almost identical (**1,487 vs 1,481 words**).

**Methodology question:** the trend groups are defined from recent performance windows and the comparison is observational. Does the evidence support "age is associated with the observed trend group" rather than "older content causes decline"? A stronger design would fix a decision date, measure age before it, and evaluate an observed future decline label afterward while grouping by client.

### Finding B — growth prediction generalization
The paper reports a **Growth Prediction** model at about **90% on unseen pages from known brands** and **75% on completely unseen brands**, with repeated random tests.

**Methodology question:** because growth is a temporal outcome, does a random page split — even with an unseen-brand variant — fully mimic deployment into the future? I would want the metric named next to the base rate and a time-forward evaluation in addition to brand grouping before treating the headline percentage as stable future performance.

These are constructive questions about claim scope, not claims that the paper is wrong.

In [ ]:
paper_findings={
    "finding_a":{
        "growing_avg_age_days":185,
        "declining_avg_age_days":228,
        "growing_avg_words":1487,
        "declining_avg_words":1481,
        "allowed_claim":"observed association; not causal"
    },
    "finding_b":{
        "growth_prediction_same_brand_pct":90,
        "growth_prediction_new_brand_pct":75,
        "audit_request":"name metric + base rate + time-forward evaluation"
    }
}
print(json.dumps(paper_findings,indent=2))


## 2. My model under an honest split (before/after)

I deliberately compare a convenient **random-row split** against the deployment-relevant **client holdout**. The model, features, seed, and Precision@50 metric stay fixed.

The point is not to maximize the random-split number. The **gap** tells me how much apparent skill may come from rows that share client-specific structure across train and test.

In [ ]:
import os, subprocess, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

REPO_URL="https://github.com/3bud-ZC/flyrank-ml-internship"
REPO_DIR="flyrank-ml-internship"

def find_repo_root():
    here=Path.cwd().resolve()
    for candidate in [here,*here.parents]:
        if (candidate/"data/raw/content_refresh_anonymized.csv").exists():
            return candidate
    return None

root=find_repo_root()
if root is None:
    if not Path(REPO_DIR).exists():
        subprocess.run(["git","clone","--depth","1",REPO_URL,REPO_DIR],check=True)
    root=Path(REPO_DIR).resolve()
os.chdir(root)

df=pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["target"]=df["trend_direction"].str.lower().eq("down").astype(int)
features=[
    "impressions_90d","clicks_90d","sessions_90d","avg_position","ctr",
    "content_age_days","days_since_last_update","word_count",
    "engagement_rate","scroll_rate","days_with_impressions","days_with_sessions"
]
X=df[features]
y=df["target"].to_numpy()

def make_rf():
    return Pipeline([
        ("imputer",SimpleImputer(strategy="median")),
        ("model",RandomForestClassifier(
            n_estimators=300,max_depth=10,min_samples_leaf=25,
            class_weight="balanced_subsample",n_jobs=-1,random_state=42
        ))
    ])

def p_at_k(labels,scores,k=50):
    order=np.argsort(-scores)[:k]
    return float(np.asarray(labels)[order].mean())

# convenient random rows
r_train,r_test=train_test_split(np.arange(len(df)),test_size=.2,random_state=42,stratify=y)
rf_random=make_rf()
rf_random.fit(X.iloc[r_train],y[r_train])
random_score=rf_random.predict_proba(X.iloc[r_test])[:,1]

# honest client holdout, same rule as Weeks 4-5
rng=np.random.default_rng(42)
clients=df["client_id"].drop_duplicates().to_numpy()
test_clients=set(rng.permutation(clients)[:max(1,int(round(len(clients)*.20)))])
g_test=np.where(df["client_id"].isin(test_clients))[0]
g_train=np.where(~df["client_id"].isin(test_clients))[0]
rf_group=make_rf()
rf_group.fit(X.iloc[g_train],y[g_train])
group_score=rf_group.predict_proba(X.iloc[g_test])[:,1]

comparison={
    "random_row_p50":p_at_k(y[r_test],random_score,50),
    "client_holdout_p50":p_at_k(y[g_test],group_score,50),
    "random_test_base_rate":float(y[r_test].mean()),
    "client_test_base_rate":float(y[g_test].mean()),
    "client_overlap":int(len(set(df.iloc[g_train]["client_id"]) & set(df.iloc[g_test]["client_id"])))
}
print(json.dumps(comparison,indent=2))


## 3. Leakage audit

I audit the final starter feature set against the three main leakage classes:

1. **Label-derived:** `trend_direction` and `trend_pct` create the starter proxy, so neither can be a feature.
2. **Future/overlapping:** the starter data is a same-snapshot exercise, so I make no future-prediction claim from it. The eventual warehouse target must place all feature dates before the label dates.
3. **Decision-derived:** product health/action/priority flags are not used as model inputs.

IDs are context only for grouping and joins. They are not predictive features.

In [ ]:
forbidden={
    "trend_direction","trend_pct","is_declining_label","is_declining_proxy",
    "health_score","priority_score","action_type","content_id","client_id"
}
present=sorted(forbidden.intersection(features))
audit_receipt={
    "feature_count":len(features),
    "forbidden_features_present":present,
    "label_siblings_excluded":True,
    "ids_as_features":False,
    "product_decisions_as_features":False,
    "client_overlap_in_grouped_split":comparison["client_overlap"],
    "future_claim_on_same_window_proxy":False
}
print(json.dumps(audit_receipt,indent=2))
assert present==[]
assert comparison["client_overlap"]==0


## 4. Claim rewrite

**Too strong:** "The model identifies the pages that should be refreshed and will improve traffic."

**Evidence-safe rewrite:** "On the bundled 30,000-row starter slice, the model ranks pages against a same-window decline proxy. Under a client-holdout evaluation it achieved the measured Precision@50 reported below. The output is a **decision-support review queue**: it identifies pages worth human inspection first; it does not show that refreshing a flagged page causes traffic recovery."

This wording separates:
- observed model performance,
- the proxy nature of the target,
- out-of-sample ranking evidence,
- and the absence of a causal design.

In [ ]:
safe_claim={
    "verbs":["observed","measured","ranked","flagged for review"],
    "causal_verbs_used":False,
    "population":"bundled 30,000-row anonymized starter slice",
    "validation":"client holdout",
    "target_scope":"same-window decline proxy",
    "action_scope":"human decision support"
}

receipt={
    "paper_findings":paper_findings,
    "before_after":comparison,
    "leakage_audit":audit_receipt,
    "safe_claim":safe_claim
}
Path("work/outputs").mkdir(parents=True,exist_ok=True)
Path("work/outputs/w06_validation_receipt.json").write_text(json.dumps(receipt,indent=2))
print(json.dumps(safe_claim,indent=2))


## Self-check

- [x] Two FlyRank research findings are reviewed constructively
- [x] Random-row and client-grouped validation are compared with the same model
- [x] Base rates are shown next to Precision@50
- [x] Leakage audit covers label siblings, future windows, decisions, and IDs
- [x] Strong claim is rewritten as observed/measured decision-support language
- [ ] Notebook executed top to bottom with visible outputs
- [ ] Validation receipt committed
- [ ] Submit the public repository URL on the ML-09 card